# Day 47：实用 Agent 场景专题（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 7 周**：Agent 工程化护栏、行业场景与多 Agent 办公助手实战  
> **GitLab 工单**：`STARMIND-047` ｜ **开发分支**：`feat/day47-text2sql-browser-code-agents`  
> **配套 HTML5 课件**：`courseware/day47/index.html` ｜ **独立版本快照**：`versions/day47/`

---

## 🎙️ 老陈开场旁白

早啊各位！如果问国内 B 端企业老板最愿意掏钱买单的 Agent 场景是什么，排前三的一定是：**Text-to-SQL 智能问数、代码研发助手、以及浏览器自动化操作（Computer Use / Browser Agent）**。但 Text-to-SQL 也是线上最容易出“删库跑路”事故的高危区——千万别把生产库高权限账号直接交给大模型！今天上半场老陈带大家建一个真实的 SQLite 销售库，亲手打造带 **只读安全防注入守卫 + 自动 LIMIT 保护 + 报错自纠错** 的 Text-to-SQL 数据分析 Agent；下半场我们拆解并编码实现浏览器 DOM 压缩规划与代码助手 Diff 补丁闭环原型！

### 承上启下
- **昨日基线**：昨天（Day 46）我们为 Agent 装上了 Trace/Span 全链路观测、Guardrails 三道安全护栏、死循环熔断和分层降本路由。今天我们把这些硬核工程武器直接投入企业落地率最高的三类垂直 Agent 场景！
- **明日预告**：明天（Day 48）正式开启第四阶段压轴大戏——「阶段项目三（上）：多 Agent 智能办公助手」，我们将完成整体多 Agent 架构设计、专家角色划分，并一口气落地 5 大企业办公工具集。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day47'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-047）

星辰数科商业分析部每天接到业务线 80+ 个临时取数需求（如“华东区 Q3 销售额最高的 3 个产品是什么？”“各区域平均客单价对比”），数据分析师疲于写重复 SQL；同时研发团队也在探索安全可控的代码自动修复与网页巡检自动化 Agent。

**今日迭代目标：**
- 构建真实 SQLite 销售数据库（含 `customers`、`products`、`sales_orders` 三张关联表），实现带 `SQLSafetyGuard` 双重只读保护的 Text-to-SQL 数据分析 Agent。
- 实现 SQL 语法/字段报错自动捕获与自纠错重试机制，并将查询结果自动格式化为业务洞察摘要。
- 实现浏览器自动化（DOM 精简快照 + 动作规划）与代码助手（代码检索 -> Unified Diff 补丁生成 -> 沙箱校验）核心架构原型。

**验收标准：**
- 自然语言提问“华东区总销售额”、“各区域销售额排名”或“最畅销商品”时，能生成正确 SQL 并在真实 SQLite 上查出精准数值。
- 任何包含 `DROP/DELETE/UPDATE/INSERT/ALTER/PRAGMA` 或多语句分号 `;` 的 SQL 均被 `SQLSafetyGuard` 100% 拦截。
- 缺少 `LIMIT` 的查询自动追加 `LIMIT 100`；代码助手原型能基于问题生成标准 Diff 并通过校验，所有单测本地 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. Text-to-SQL 生产防注入的“双重锁”原则

千万不要只依赖应用层正则表达式去拦 SQL 注入！因为 SQL 语法变种极多（注释混淆、CTE 嵌套、特殊内置函数）。生产标准必须是**双锁叠加**：
1) 应用层 `SQLSafetyGuard` 拦截非 `SELECT`、多语句分号、危险关键词并强制追加 `LIMIT`；
2) 数据库层配置**只读数据库账号（仅授予目标表 `SELECT` 权限）** + SQLite `PRAGMA query_only = ON` + 语句超时（`statement_timeout = 5s`）。

> **⚠️ 老陈避坑**：坑点：在校验多语句注入时，记得先去掉 SQL 末尾合法的单个分号 `;` 再检查中间是否还有 `;`，否则正常带结尾分号的 `SELECT ...;` 会被误杀！

### 2. 2. 业务口径词典（Semantic Layer）决定 Text-to-SQL 准确率上限

企业问数最容易出错的往往不是 SQL 语法，而是**业务口径**！例如“有效销售额”到底包不包含 `status = 'CANCELLED'` 的退单？必须在 Schema Prompt 中明确注入业务口径（如“统计销售额时必须过滤 `status = 'PAID'`”），否则模型算出来的数和财务报表永远对不上。

> **⚠️ 老陈避坑**：坑点：直接把几百张裸表塞进 Prompt 会导致模型选错表；大型数仓应先建立“指标宽表/视图（View）”，让 Agent 只查干净的业务视图。

### 3. 3. Browser Agent 与 Coding Agent 的核心工程共性

无论是操作网页还是修改代码，高成功率的秘诀都是**压缩观测空间 + 原子化动作 + 即时反馈验证**：浏览器压缩为编号可交互树，代码压缩为目标函数片段与 Unified Diff，每步执行后立即获取页面状态或单测结果作为下一轮 `Observation`。

> **⚠️ 老陈避坑**：坑点：让 Coding Agent 直接在生产代码仓无沙箱运行 `subprocess` 执行任意命令是重大安全隐患，必须在隔离容器或受限只读测试环境中执行校验。


## 💻 实操步骤 1：步骤一：连接真实 SQLite 销售库的只读安全 Text-to-SQL 数据分析 Agent

- **目标文件**：`versions/day47/src/day47_text2sql_agent.py`
- **老陈代码旁白**：我们直接用 Python 内置 `sqlite3` 建一个真实的星辰数科销售库（包含客户、产品、订单三张表并灌入真实测试数据），然后给它套上应用层 `SQLSafetyGuard` + 数据库层 `PRAGMA query_only = ON` 双重只读安全锁，跑通从自然语言问数、自动加 LIMIT、报错自修复到生成分析结论的完整链路！


In [ ]:
"""Day 47 - 步骤一：连接真实 SQLite 销售库的只读安全 Text-to-SQL 数据分析 Agent。"""

from __future__ import annotations

import re
import sqlite3
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union


class SQLSafetyError(ValueError):
    """当候选 SQL 违反只读安全策略或疑似注入攻击时抛出。"""


class SQLSafetyGuard:
    """应用层 SQL 只读安全守卫：仅允许单条 SELECT 查询，自动补齐 LIMIT。"""

    FORBIDDEN_KEYWORDS = re.compile(
        r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|CREATE|REPLACE|TRUNCATE|ATTACH|DETACH|PRAGMA|EXEC|UNION\s+ALL\s+SELECT\s+sqlite_master)\b",
        re.IGNORECASE,
    )
    COMMENT_PATTERN = re.compile(r"(--|/\*|\*/)")

    def __init__(self, default_limit: int = 100) -> None:
        self.default_limit = default_limit

    def sanitize_and_validate(self, sql: str) -> str:
        cleaned = sql.strip()
        if cleaned.endswith(";"):
            cleaned = cleaned[:-1].strip()

        if not cleaned:
            raise SQLSafetyError("SQL 不能为空")
        if ";" in cleaned:
            raise SQLSafetyError("安全拦截：禁止多语句分号注入")
        if self.COMMENT_PATTERN.search(cleaned):
            raise SQLSafetyError("安全拦截：禁止 SQL 注释符绕过")
        if not re.match(r"^(SELECT|WITH)\b", cleaned, re.IGNORECASE):
            raise SQLSafetyError("安全拦截：仅允许执行只读 SELECT / WITH 查询")
        if self.FORBIDDEN_KEYWORDS.search(cleaned):
            raise SQLSafetyError("安全拦截：检测到写操作或高危系统关键字")

        # 自动补齐 LIMIT 保护，防止全表扫描撑爆内存
        if not re.search(r"\bLIMIT\s+\d+\b", cleaned, re.IGNORECASE):
            cleaned = f"{cleaned} LIMIT {self.default_limit}"
        return cleaned


def init_sales_sqlite_db(db_path: Union[str, Path] = ":memory:") -> sqlite3.Connection:
    """创建并初始化星辰数科真实 SQLite 销售数据库，初始化完成后开启只读模式。"""
    conn = sqlite3.connect(str(db_path))
    conn.executescript(
        """
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT NOT NULL,
            category TEXT NOT NULL,
            unit_price REAL NOT NULL
        );
        CREATE TABLE sales_orders (
            order_id INTEGER PRIMARY KEY,
            region TEXT NOT NULL,
            quarter TEXT NOT NULL,
            product_id INTEGER NOT NULL,
            quantity INTEGER NOT NULL,
            amount REAL NOT NULL,
            status TEXT NOT NULL,
            FOREIGN KEY(product_id) REFERENCES products(product_id)
        );
        INSERT INTO products VALUES
            (1, '星辰智脑企业版', 'AI平台', 50000.0),
            (2, 'RAG知识库套件', 'AI应用', 20000.0),
            (3, '多Agent办公助手', 'AI应用', 30000.0);
        INSERT INTO sales_orders VALUES
            (101, '华东', '2025Q3', 1, 3, 150000.0, 'PAID'),
            (102, '华东', '2025Q3', 3, 4, 120000.0, 'PAID'),
            (103, '华东', '2025Q3', 2, 2, 40000.0, 'CANCELLED'),
            (104, '华南', '2025Q3', 1, 2, 100000.0, 'PAID'),
            (105, '华南', '2025Q3', 2, 5, 100000.0, 'PAID'),
            (106, '华北', '2025Q3', 3, 3, 90000.0, 'PAID');
        """
    )
    conn.commit()
    # 数据库引擎级物理只读锁（第二道防线）
    conn.execute("PRAGMA query_only = ON;")
    return conn


class TextToSQLAgent:
    """只读安全 Text-to-SQL 数据分析 Agent（含报错自修复重试与业务洞察生成）。"""

    def __init__(self, conn: sqlite3.Connection, default_limit: int = 100) -> None:
        self.conn = conn
        self.guard = SQLSafetyGuard(default_limit=default_limit)

    def get_schema_summary(self) -> str:
        rows = self.conn.execute(
            "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
        ).fetchall()
        return "\n".join(f"{name}: {ddl}" for name, ddl in rows)

    def _plan_sql(self, question: str, prev_error: str = "") -> str:
        """本地确定性 SQL 规划器（生产环境替换为对接大模型的 SQL 生成节点）。"""
        # 模拟常见注入攻击语句透传，交由 SQLSafetyGuard 统一拦截验证
        if any(k in question.upper() for k in ("DROP ", "DELETE ", "UPDATE ", ";", "--", "PRAGMA ")):
            return question

        # 演示自纠错：首次故意写错列名 total_amt，捕获 OperationalError 后二次修正为 amount
        if "各区域" in question and not prev_error and "模拟纠错" in question:
            return "SELECT region, SUM(total_amt) AS total_sales FROM sales_orders WHERE status = 'PAID' GROUP BY region"

        if "各区域" in question or "区域" in question:
            return (
                "SELECT region, SUM(amount) AS total_sales, COUNT(*) AS order_cnt "
                "FROM sales_orders WHERE status = 'PAID' "
                "GROUP BY region ORDER BY total_sales DESC"
            )
        if "最畅销" in question or "商品" in question or "产品" in question:
            return (
                "SELECT p.product_name, SUM(o.amount) AS total_sales, SUM(o.quantity) AS total_qty "
                "FROM sales_orders o JOIN products p ON o.product_id = p.product_id "
                "WHERE o.status = 'PAID' GROUP BY p.product_name ORDER BY total_sales DESC"
            )
        return "SELECT COUNT(*) AS paid_orders, SUM(amount) AS total_sales FROM sales_orders WHERE status = 'PAID'"

    def analyze(self, question: str, max_retries: int = 2) -> Dict[str, Any]:
        prev_error = ""
        retries = 0
        for attempt in range(max_retries + 1):
            candidate_sql = self._plan_sql(question, prev_error=prev_error)
            safe_sql = self.guard.sanitize_and_validate(candidate_sql)
            try:
                cur = self.conn.execute(safe_sql)
                columns = [desc[0] for desc in cur.description or []]
                rows = [dict(zip(columns, r)) for r in cur.fetchall()]
                return {
                    "question": question,
                    "sql": safe_sql,
                    "row_count": len(rows),
                    "rows": rows,
                    "retries": retries,
                    "insight": self._format_insight(question, rows),
                }
            except sqlite3.OperationalError as exc:
                prev_error = str(exc)
                retries += 1
                if attempt >= max_retries:
                    raise RuntimeError(f"SQL 执行连续失败: {prev_error}") from exc
        raise RuntimeError("未预期的执行分支")

    @staticmethod
    def _format_insight(question: str, rows: List[Dict[str, Any]]) -> str:
        if not rows:
            return f"针对「{question}」，未查询到符合条件的有效已支付订单记录。"
        top = rows[0]
        summary_pairs = ", ".join(f"{k}={v}" for k, v in top.items())
        return f"共检索到 {len(rows)} 条聚合结果，排首位的核心数据为：{summary_pairs}。"


if __name__ == "__main__":
    conn = init_sales_sqlite_db()
    agent = TextToSQLAgent(conn)
    res1 = agent.analyze("请统计各区域有效销售额排名（模拟纠错）")
    print("SQL:", res1["sql"])
    print("自纠错次数:", res1["retries"], "| 洞察:", res1["insight"])
    print("数据明细:", res1["rows"])
    conn.close()


### 🔍 步骤 1 代码深度解析

1. `init_sales_sqlite_db` 在建表灌数据后立即执行 `PRAGMA query_only = ON;`，即便上层守卫被绕过，SQLite 引擎本身也会拒绝一切写操作。
2. `SQLSafetyGuard` 拦截多语句分号、注释符与高危写关键字，并自动为没有 `LIMIT` 的查询追加 `LIMIT 100`。
3. `analyze` 完整演示了 SQL 报错捕获 -> 带着 `prev_error` 重新生成合法 SQL 的自纠错闭环（已支付订单严格过滤了 `CANCELLED` 退单）。

**预期运行输出参考：**
```text
SQL: SELECT region, SUM(amount) AS total_sales, COUNT(*) AS order_cnt FROM sales_orders WHERE status = 'PAID' GROUP BY region ORDER BY total_sales DESC LIMIT 100
自纠错次数: 1 | 洞察: 共检索到 3 条聚合结果，排首位的核心数据为：region=华东, total_sales=270000.0, order_cnt=2。
数据明细: [{'region': '华东', 'total_sales': 270000.0, 'order_cnt': 2}, {'region': '华南', 'total_sales': 200000.0, 'order_cnt': 2}, {'region': '华北', 'total_sales': 90000.0, 'order_cnt': 1}]
```


## 💻 实操步骤 2：步骤二：浏览器自动化 DOM 压缩规划器与代码助手 Unified Diff 补丁引擎

- **目标文件**：`versions/day47/src/day47_browser_code_agent.py`
- **老陈代码旁白**：下半场我们把另外两大热门场景——Browser Agent 与 Coding Agent 的核心引擎骨架亲手落地：前者将杂乱 DOM 树压缩为紧凑的编号可交互元素并规划操作序列；后者按「检索 -> 生成最小 Unified Diff -> 应用并断言验证」闭环安全修复代码。


In [ ]:
"""Day 47 - 步骤二：浏览器 DOM 压缩规划器与代码助手 Unified Diff 补丁验证原型。"""

from __future__ import annotations

import difflib
from dataclasses import dataclass
from typing import Any, Dict, List


@dataclass
class DOMElement:
    elem_id: int
    tag: str
    role: str
    text: str
    visible: bool = True
    interactive: bool = False


class BrowserDOMPlanner:
    """浏览器自动化核心：将原始 DOM 压缩为紧凑可交互快照，再规划原子操作序列。"""

    def compress_dom(self, raw_nodes: List[DOMElement]) -> List[Dict[str, Any]]:
        """过滤不可见与纯装饰节点，只保留可交互元素，通常可削减 85%+ Token。"""
        return [
            {"id": n.elem_id, "tag": n.tag, "role": n.role, "label": n.text}
            for n in raw_nodes
            if n.visible and n.interactive
        ]

    def plan_actions(self, goal: str, compact_dom: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        actions: List[Dict[str, Any]] = []
        search_box = next((e for e in compact_dom if e["tag"] == "input"), None)
        submit_btn = next((e for e in compact_dom if e["tag"] == "button" and "查询" in e["label"]), None)
        if search_box:
            keyword = goal.replace("查询", "").strip() or "星辰智脑"
            actions.append({"action": "type", "target_id": search_box["id"], "value": keyword})
        if submit_btn:
            actions.append({"action": "click", "target_id": submit_btn["id"]})
        return actions


class CodingAssistantPrototype:
    """代码助手核心：最小化 Unified Diff 补丁生成与沙箱函数验证闭环。"""

    def generate_unified_diff(self, file_name: str, original: str, patched: str) -> str:
        diff_lines = difflib.unified_diff(
            original.splitlines(keepends=True),
            patched.splitlines(keepends=True),
            fromfile=f"a/{file_name}",
            tofile=f"b/{file_name}",
        )
        return "".join(diff_lines)

    def fix_zero_division_bug(self, source_code: str) -> Dict[str, Any]:
        """定位除零隐患并生成带防御判断的最小化补丁与 Unified Diff。"""
        buggy_line = "return total / count"
        fixed_line = "return total / count if count else 0.0"
        if buggy_line not in source_code:
            return {"changed": False, "patched_code": source_code, "diff": ""}

        patched = source_code.replace(buggy_line, fixed_line, 1)
        diff_text = self.generate_unified_diff("metrics.py", source_code, patched)
        return {"changed": True, "patched_code": patched, "diff": diff_text}


if __name__ == "__main__":
    raw_dom = [
        DOMElement(1, "div", "banner", "装饰横幅", visible=True, interactive=False),
        DOMElement(2, "input", "textbox", "输入客户名称", visible=True, interactive=True),
        DOMElement(3, "button", "button", "查询客户", visible=True, interactive=True),
        DOMElement(4, "button", "button", "隐藏导出", visible=False, interactive=True),
    ]
    planner = BrowserDOMPlanner()
    compact = planner.compress_dom(raw_dom)
    print("压缩后 DOM 元素数:", len(compact), "| 规划动作:", planner.plan_actions("查询 华东客户", compact))

    buggy_src = "def avg_order(total: float, count: int) -> float:\n    return total / count\n"
    coder = CodingAssistantPrototype()
    fix = coder.fix_zero_division_bug(buggy_src)
    print("代码助手生成 Diff:\n" + fix["diff"])


### 🔍 步骤 2 代码深度解析

1. `BrowserDOMPlanner.compress_dom` 过滤掉隐藏节点和纯布局 `div`，只把可交互元素送入规划器，这是真实 Browser Agent 提速降本的关键。
2. `CodingAssistantPrototype` 使用标准库 `difflib.unified_diff` 输出工业标准的 Git Diff 补丁，方便人工在 MR 中逐行审阅而不破坏文件其他注释。

**预期运行输出参考：**
```text
压缩后 DOM 元素数: 2 | 规划动作: [{'action': 'type', 'target_id': 2, 'value': '华东客户'}, {'action': 'click', 'target_id': 3}]
代码助手生成 Diff:
--- a/metrics.py
+++ b/metrics.py
@@ -1,2 +1,2 @@
 def avg_order(total: float, count: int) -> float:
-    return total / count
+    return total / count if count else 0.0

```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day47/tests/test_day47.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 47 自动化单元测试：验证真实 SQLite 销售库 Text-to-SQL、防注入拦截及浏览器/代码助手原型。"""

import pytest
from src.day47_text2sql_agent import SQLSafetyError, TextToSQLAgent, init_sales_sqlite_db
from src.day47_browser_code_agent import BrowserDOMPlanner, CodingAssistantPrototype, DOMElement


def test_text2sql_queries_real_sqlite_and_self_corrects():
    conn = init_sales_sqlite_db()
    agent = TextToSQLAgent(conn, default_limit=50)

    res_region = agent.analyze("统计各区域有效销售额（模拟纠错）")
    assert res_region["retries"] == 1
    assert "LIMIT 50" in res_region["sql"]
    # 华东有效已支付订单应为 150000 + 120000 = 270000（不含 40000 的 CANCELLED 退单）
    assert res_region["rows"][0]["region"] == "华东"
    assert res_region["rows"][0]["total_sales"] == 270000.0

    res_prod = agent.analyze("最畅销的商品是什么")
    assert res_prod["rows"][0]["product_name"] == "星辰智脑企业版"
    assert res_prod["rows"][0]["total_sales"] == 250000.0
    conn.close()


@pytest.mark.parametrize(
    "malicious_sql",
    [
        "DROP TABLE sales_orders",
        "DELETE FROM sales_orders WHERE order_id = 101",
        "SELECT * FROM products; DROP TABLE products",
        "SELECT * FROM products -- 注释绕过",
        "PRAGMA query_only = OFF",
    ],
)
def test_text2sql_blocks_injection_and_writes(malicious_sql: str):
    conn = init_sales_sqlite_db()
    agent = TextToSQLAgent(conn)
    with pytest.raises(SQLSafetyError):
        agent.analyze(malicious_sql)
    conn.close()


def test_browser_planner_and_coding_assistant():
    planner = BrowserDOMPlanner()
    compact = planner.compress_dom(
        [
            DOMElement(1, "div", "text", "噪音", visible=True, interactive=False),
            DOMElement(2, "input", "textbox", "搜索框", visible=True, interactive=True),
            DOMElement(3, "button", "button", "查询", visible=True, interactive=True),
        ]
    )
    assert len(compact) == 2
    actions = planner.plan_actions("查询 订单101", compact)
    assert [a["action"] for a in actions] == ["type", "click"]

    coder = CodingAssistantPrototype()
    res = coder.fix_zero_division_bug("def avg(total, count):\n    return total / count\n")
    assert res["changed"] is True
    assert "-    return total / count" in res["diff"]
    assert "+    return total / count if count else 0.0" in res["diff"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 Text-to-SQL 增加表名白名单校验

**题目要求：**
编写 `check_allowed_tables(sql, allowed_tables)`，解析 `FROM` 与 `JOIN` 后紧跟的表名，拒绝查询不在白名单内的敏感表（如 `hr_salaries`、`sqlite_master`）。

**思路提示：** 使用正则 `re.findall(r'\b(?:FROM|JOIN)\s+([a-zA-Z0-9_]+)', sql, re.I)` 提取表名并检查是否全部属于 `allowed_tables`。


In [ ]:
"""Day 47 作业 1 参考答案：Text-to-SQL 表名白名单校验器。"""

import re
from typing import Set


def check_allowed_tables(sql: str, allowed_tables: Set[str]) -> bool:
    tables = re.findall(r"\b(?:FROM|JOIN)\s+([a-zA-Z0-9_]+)", sql, re.IGNORECASE)
    return all(t.lower() in {a.lower() for a in allowed_tables} for t in tables)


if __name__ == "__main__":
    allowed = {"products", "sales_orders"}
    assert check_allowed_tables("SELECT * FROM sales_orders JOIN products ON 1=1", allowed) is True
    assert check_allowed_tables("SELECT * FROM hr_salaries", allowed) is False
    print("表名白名单校验通过")


**答案解析与评分标准：**

表级白名单能防止即便是只读账号也误查到同一库内的薪酬、密钥等高敏系统表。


### 作业 2（企业实战）：实现 Text-to-SQL 结果集行数截断与列脱敏

**题目要求：**
编写 `mask_sensitive_columns(rows, sensitive_cols={'phone', 'email'})`，对查询结果字典列表中的敏感列自动替换为 `***`，避免业务问数泄露客户隐私。

**思路提示：** 遍历每行字典，对命中 `sensitive_cols` 的键赋值为 `'***'`。


In [ ]:
"""Day 47 作业 2 参考答案：SQL 结果集敏感列自动脱敏。"""

from typing import Any, Dict, List, Set


def mask_sensitive_columns(rows: List[Dict[str, Any]], sensitive_cols: Set[str]) -> List[Dict[str, Any]]:
    return [
        {k: ("***" if k.lower() in sensitive_cols else v) for k, v in row.items()}
        for row in rows
    ]


if __name__ == "__main__":
    raw = [{"customer": "星辰客户A", "phone": "13800138000", "amount": 50000}]
    safe = mask_sensitive_columns(raw, {"phone", "email"})
    print(safe)
    assert safe[0]["phone"] == "***" and safe[0]["amount"] == 50000


**答案解析与评分标准：**

在 SQL 返回层做列级自动脱敏，与 Day 46 的文本 PII 护栏形成互补。


### 作业 3（架构进阶）：为 Coding Agent 增加统一补丁应用前的语法预检

**题目要求：**
编写 `verify_python_syntax(code_str)`，利用内置 `ast.parse` 在写入文件前校验补丁后的 Python 代码是否存在语法错误，返回 `(ok, error_msg)`。

**思路提示：** 捕获 `SyntaxError` 并返回具体出错行号与提示。


In [ ]:
"""Day 47 作业 3 参考答案：Coding Agent AST 语法预检器。"""

import ast
from typing import Tuple


def verify_python_syntax(code_str: str) -> Tuple[bool, str]:
    try:
        ast.parse(code_str)
        return True, "OK"
    except SyntaxError as exc:
        return False, f"第 {exc.lineno} 行语法错误: {exc.msg}"


if __name__ == "__main__":
    assert verify_python_syntax("x = 1 + 2")[0] is True
    ok, msg = verify_python_syntax("def broken(:\n    pass")
    print("语法拦截:", ok, msg)
    assert ok is False


**答案解析与评分标准：**

AST 语法预检耗时不到 1 毫秒，却能在落盘前拦掉绝大多数大模型漏括号、缩进错位的低级补丁。


---

## 🎯 老陈收尾复盘

今天这一仗打得太爽了！大家亲眼看到：真实 SQLite 销售库配合 `SQLSafetyGuard`（只读白名单 + 禁多语句分号 + 强 LIMIT 截断 + `PRAGMA query_only=ON` 双重物理锁），既能让运营总监用大白话秒查各区域销售额和 Top 商品，又能把恶意 `DROP TABLE` 注入死死挡在门外。明天我们正式启动阶段项目三「多 Agent 智能办公助手」，准备大干一场！
